# Agent Sheep (Phase 1: No MQTT)

This notebook runs a minimal sheep-only grid simulation using fixed tick order:
move → eat → reproduce → die → regrow grass.

It uses `simulated_city.config.load_config()` for reproducible seed selection and keeps all logic local (no MQTT in Phase 1).

In [ ]:
# Cell purpose: import helpers and load project configuration.
from simulated_city.config import load_config
from simulated_city.sheep_wolf_models import SheepSimulationParams, run_simulation

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

In [ ]:
# Cell purpose: define Phase 1 simulation parameters and deterministic seed.
params = SheepSimulationParams(
    grid_width=10,
    grid_height=10,
    initial_sheep=30,
    initial_sheep_energy=8,
    sheep_move_cost=1,
    sheep_eat_gain=4,
    sheep_reproduction_threshold=10,
    sheep_reproduction_probability=0.08,
    grass_regrow_ticks=5,
)

seed = 42
if config.simulation is not None and config.simulation.seed is not None:
    seed = int(config.simulation.seed)

tick_count = 30
print(f"Using seed={seed}, ticks={tick_count}, grid={params.grid_width}x{params.grid_height}")

In [ ]:
# Cell purpose: run the local sheep simulation and print per-tick summaries.
summaries = run_simulation(params=params, ticks=tick_count, seed=seed)

print("tick sheep grass births deaths ate avg_energy")
for summary in summaries:
    print(
        f"{summary.tick:>4} {summary.sheep_count:>5} {summary.grass_grown_cells:>5} "
        f"{summary.births:>6} {summary.deaths:>6} {summary.sheep_ate_grass:>3} "
        f"{summary.average_energy:>10.2f}"
    )

final_summary = summaries[-1]
print()
print("Final summary:")
print(
    {
        "tick": final_summary.tick,
        "sheep_count": final_summary.sheep_count,
        "grass_grown_cells": final_summary.grass_grown_cells,
        "average_energy": round(final_summary.average_energy, 3),
    }
)

In [ ]:
# Cell purpose: verify reproducibility by running the same seed twice.
first_run = run_simulation(params=params, ticks=tick_count, seed=seed)
second_run = run_simulation(params=params, ticks=tick_count, seed=seed)

same = [
    (
        a.tick == b.tick
        and a.sheep_count == b.sheep_count
        and a.grass_grown_cells == b.grass_grown_cells
        and a.births == b.births
        and a.deaths == b.deaths
        and a.sheep_ate_grass == b.sheep_ate_grass
        and round(a.average_energy, 8) == round(b.average_energy, 8)
    )
    for a, b in zip(first_run, second_run)
]

print(f"Deterministic run check: {all(same)}")